In [1]:
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Executable: /usr/bin/python3


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    properties = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(
            properties.total_memory / (1024 ** 3),
            2,
        ),
        "GB",
    )
else:
    print("No CUDA GPU detected.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [3]:
from pathlib import Path

WORKSPACE = Path("/content")
REPO_NAME = "IOAI-2026-Challenge-Solutions"
REPO_DIR = WORKSPACE / REPO_NAME

print("Repo target:", REPO_DIR)

Repo target: /content/IOAI-2026-Challenge-Solutions


In [4]:
import subprocess


def run_command(command: list[str], cwd: str | None = None) -> None:
    print("$", " ".join(command))

    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Command failed with code {result.returncode}"
        )

In [5]:
if REPO_DIR.exists():
    print("Repository already exists.")
else:
    run_command(
        [
            "git",
            "clone",
            "https://github.com/Aihan-2005/IOAI-2026-Challenge-Solutions.git",
            str(REPO_DIR),
        ]
    )

$ git clone https://github.com/Aihan-2005/IOAI-2026-Challenge-Solutions.git /content/IOAI-2026-Challenge-Solutions


In [6]:
run_command(
    [
        "git",
        "checkout",
        "task1-find-the-order",
    ],
    cwd=str(REPO_DIR),
)

$ git checkout task1-find-the-order
Branch 'task1-find-the-order' set up to track remote branch 'task1-find-the-order' from 'origin'.



In [7]:
run_command(
    [
        "git",
        "status",
        "--short",
        "--branch",
    ],
    cwd=str(REPO_DIR),
)

$ git status --short --branch
## task1-find-the-order...origin/task1-find-the-order



In [8]:
TASK_DIR = (
    REPO_DIR
    / "task1-find-the-order"
)

print("Repository:", REPO_DIR)
print("Task:", TASK_DIR)

assert TASK_DIR.is_dir()

print("Task directory OK")

Repository: /content/IOAI-2026-Challenge-Solutions
Task: /content/IOAI-2026-Challenge-Solutions/task1-find-the-order
Task directory OK


In [9]:
import sys

task_path = str(TASK_DIR)

if task_path not in sys.path:
    sys.path.insert(0, task_path)

print(sys.path[0])

/content/IOAI-2026-Challenge-Solutions/task1-find-the-order


In [10]:
from src.order_utils import (
    order_to_rank,
    rank_to_order,
    pairwise_score,
)

from src.predictors import (
    PrefixIndexBaseline,
)

print("Project imports OK")

Project imports OK


In [1]:
order = [1, 2, 0]

rank = order_to_rank(order)

print("Order:", order)
print("Rank:", rank)
print(
    "Round trip:",
    rank_to_order(rank),
)

NameError: name 'order_to_rank' is not defined

In [3]:
import importlib.util


packages = [
    "torch",
    "transformers",
    "datasets",
    "librosa",
    "soundfile",
]

for package in packages:
    available = (
        importlib.util.find_spec(package)
        is not None
    )

    print(
        f"{package:<15}",
        "OK" if available else "MISSING",
    )

torch           OK
transformers    OK
datasets        OK
librosa         OK
soundfile       OK


In [4]:
%pip install -q \
    transformers \
    accelerate \
    datasets \
    librosa \
    soundfile \
    huggingface_hub

In [ ]:
import torch
import transformers
import librosa
import datasets

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Librosa:", librosa.__version__)

print(
    "CUDA:",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

In [2]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [5]:
!nvidia-smi

Thu Aug 20 17:57:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import sys
print(sys.executable)

/usr/bin/python3


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/1_Find_the_Order/code/baseline/solution.ipynb)

# Find the Order — baseline on Google Colab

Run all (GPU runtime recommended). The first cell fetches the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-find-the-order) and recreates the contest file layout; every cell after it is the original, untouched baseline.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install librosa soundfile huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-find-the-order", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)
# merged dataset root: split folders + *_answers files, public and private together
DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)
print("dataset root:", DSROOT, "->", sorted(p.name for p in DSROOT.iterdir()))

EVAL_SPLIT = "test_leaderboard_a"   # which graded split plays the role of dataset/test_public
ws = Path.cwd()
for name in ("train","pretrain","pretest"): link(DSROOT/name, ws/"dataset"/name)
link(DSROOT/EVAL_SPLIT, ws/"dataset"/"test_public")
# the three frozen model bundles the contest shipped offline
for repo, name in [("facebook/wav2vec2-base-960h","wav2vec2-base-960h"),
                   ("openai/whisper-small","whisper-small"),
                   ("Qwen/Qwen2.5-0.5B","qwen2.5-0.5b")]:
    p = Path("models")/name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        link(snapshot_download(repo), p)
print("ready: dataset/ (test_public ->", EVAL_SPLIT + ") and models/")


# Find the Order — Baseline solution

A **deterministic** baseline. It uses `prefix.json` — the two chunks known to come
first — and leaves every other chunk in the order it was given.

It writes **`answers.json`** at the repo root in the rank convention: `P[i]` is the
predicted chronological position of `chunk_i.wav`.

Replace the logic below with your own.

In [ ]:
import os, json

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

# prefix.json ships with every split. answers.json does NOT exist in the hidden
# grading set -- never read it from TEST_DIR, or your notebook dies at grade time.
with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefix = json.load(f)

def n_chunks(ddir):
    return sum(1 for f in os.listdir(ddir)
               if f.startswith("chunk_") and f.endswith(".wav"))

# Only numeric directories are dialogues -- skip strays such as .ipynb_checkpoints,
# which JupyterLab creates as soon as you open something inside the folder.
dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR)
     if d.isdigit() and os.path.isdir(os.path.join(TEST_DIR, d))),
    key=int,
)

answers = {}
for did in dialogue_ids:
    n = n_chunks(os.path.join(TEST_DIR, did))
    first, second = prefix[did]
    order = [first, second] + [i for i in range(n) if i not in (first, second)]
    rank = [0] * n                      # rank[i] = position of chunk_i
    for pos, idx in enumerate(order):
        rank[idx] = pos
    answers[did] = rank

with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")


## Additional Models & Libraries

## wav2-vec2-base-960h

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor

WAV2VEC_PATH = "models/wav2vec2-base-960h"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)
model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH).to(device).eval()

audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    h = model(inputs.input_values.to(device)).last_hidden_state
embed = h.mean(dim=1).squeeze().cpu().numpy()

print(f"Wav2vec embedding shape: {embed.shape}")

## Qwen2.5-0.5B

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN25_MODEL_PATH = "models/qwen2.5-0.5b"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(QWEN25_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(QWEN25_MODEL_PATH, dtype=torch.bfloat16).to(device).eval()

prompt = "The weather today is"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Whisper-small

In [ ]:
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_MODEL_PATH = "models/whisper-small"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_PATH).to(device).eval()


audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
with torch.no_grad():
    ids = model.generate(feats.to(device), language="en", task="transcribe")
result = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
print(f"Whisper ASR trasncription: {result}")